In [69]:
import pandas as pd

In [70]:
df = pd.read_csv("../data/raw/diabetic_data.csv")

In [71]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [72]:
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

# Null Handling & Type Handling

## Target Column

In [74]:
df["readmitted"].isna().sum()

np.int64(0)

In [75]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [76]:
def getLabel(x):
    if x == "<30":
        return 0
    elif x == ">30":
        return 1
    else:
        return 2

In [77]:
df["readmitted"] = df["readmitted"].apply(lambda x: getLabel(x)) 

In [78]:
target = df["readmitted"].values

## Numerical Columns

In [79]:
numerical_columns = ["time_in_hospital", "num_lab_procedures", "num_medications", "number_diagnoses"]

In [80]:
df[numerical_columns].isna().sum()

time_in_hospital      0
num_lab_procedures    0
num_medications       0
number_diagnoses      0
dtype: int64

In [81]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_input = scaler.fit_transform(df[numerical_columns])

In [82]:
numerical_input.shape

(101766, 4)

## Categorical Columns

In [83]:
categorical_columns = ["gender", "race", "admission_type_id", "discharge_disposition_id"]

In [84]:
df[categorical_columns].isna().sum()

gender                      0
race                        0
admission_type_id           0
discharge_disposition_id    0
dtype: int64

In [85]:
most_freq_race = df["race"].mode().values[0]
df["race"] = df["race"].replace({"?": most_freq_race})
df["race"].value_counts()

race
Caucasian          78372
AfricanAmerican    19210
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [86]:
df["admission_type_id"] = df["admission_type_id"].astype("object")

In [87]:
df["admission_type_id"].value_counts()

admission_type_id
1    53990
3    18869
2    18480
6     5291
5     4785
8      320
7       21
4       10
Name: count, dtype: int64

In [88]:
df["discharge_disposition_id"] = df["discharge_disposition_id"].astype("object")

In [89]:
df["discharge_disposition_id"].value_counts()

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
16       11
19        8
10        6
27        5
12        3
20        2
Name: count, dtype: int64

In [90]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output = False, handle_unknown = "ignore")
categorical_input = encoder.fit_transform(df[categorical_columns])

In [91]:
categorical_input.shape

(101766, 42)

In [92]:
categorical_input.dtype

dtype('float64')

## Textual Columns

In [93]:
textual_columns = ["diag_1", "diag_2", "diag_3"]

In [94]:
def map_icd9_to_category(code):
    try:
        code = str(code).strip()

        # V and E codes → Other
        if code.startswith('V') or code.startswith('E'):
            return 'Other'

        code_num = float(code)

        if 390 <= code_num <= 459 or code_num == 785:
            return 'Circulatory'
        elif 250 <= code_num < 251:
            return 'Diabetes'
        elif 460 <= code_num <= 519 or code_num == 786:
            return 'Respiratory'
        elif 520 <= code_num <= 579 or code_num == 787:
            return 'Digestive'
        elif 800 <= code_num <= 999:
            return 'Injury'
        elif 710 <= code_num <= 739:
            return 'Musculoskeletal'
        elif 140 <= code_num <= 239:
            return 'Neoplasms'
        elif 580 <= code_num <= 629 or code_num == 788:
            return 'Genitourinary'
        else:
            return 'Other'

    except:
        return 'Other'

In [95]:
df['diag_1'] = df['diag_1'].apply(map_icd9_to_category)
df['diag_2'] = df['diag_2'].apply(map_icd9_to_category)
df['diag_3'] = df['diag_3'].apply(map_icd9_to_category)

In [96]:
df['diagnosis_text'] = (df['diag_1'] + ' ' +
                        df['diag_2'] + ' ' +
                        df['diag_3'])

In [97]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer

In [98]:
tokenizer = Tokenizer(oov_token = "<nothing>")

In [99]:
tokenizer.fit_on_texts(df["diagnosis_text"].values)

In [100]:
tokenizer.word_index

{'<nothing>': 1,
 'circulatory': 2,
 'other': 3,
 'diabetes': 4,
 'respiratory': 5,
 'genitourinary': 6,
 'digestive': 7,
 'injury': 8,
 'musculoskeletal': 9,
 'neoplasms': 10}

In [101]:
tokenizer.word_counts

OrderedDict([('diabetes', 38708),
             ('other', 75722),
             ('circulatory', 92624),
             ('neoplasms', 7836),
             ('respiratory', 32676),
             ('injury', 11348),
             ('genitourinary', 20173),
             ('musculoskeletal', 8636),
             ('digestive', 17575)])

In [102]:
tokenizer.document_count

101766

In [103]:
sequences = tokenizer.texts_to_sequences(df["diagnosis_text"].values)

In [104]:
import numpy as np

textual_input = np.array(sequences)

## Sequential Columns

In [105]:
sequential_columns = ["metformin", "insulin", "glipizide", "glyburide", "pioglitazone"]

In [106]:
med_map = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': -1}

In [107]:
df[sequential_columns] = df[sequential_columns].replace(med_map)

C:\Users\yogip\AppData\Local\Temp\ipykernel_10992\1583903410.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[sequential_columns] = df[sequential_columns].replace(med_map)


In [108]:
sequential_input = df[sequential_columns].values.reshape(-1, 5, 1)

In [109]:
sequential_input.shape

(101766, 5, 1)

In [110]:
sequential_input.dtype

dtype('int64')

# Saving the object & Splitting the data

In [111]:
import pickle

In [112]:
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [113]:
with open("../models/encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

In [114]:
with open("../models/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [115]:
from sklearn.model_selection import train_test_split
random_state = 44

In [116]:
X_train_num, X_test_num = train_test_split(numerical_input, test_size = 0.25, random_state = random_state)

In [117]:
X_train_cat, X_test_cat = train_test_split(categorical_input, test_size = 0.25, random_state = random_state)

In [118]:
X_train_text, X_test_text = train_test_split(textual_input, test_size = 0.25, random_state = random_state)

In [119]:
X_train_seq, X_test_seq = train_test_split(sequential_input, test_size = 0.25, random_state = random_state)

In [120]:
y_train, y_test = train_test_split(target, test_size = 0.25, random_state = random_state)

In [121]:
with open("../data/processed/X_train_num.pkl", "wb") as f:
    pickle.dump(X_train_num, f)

In [122]:
with open("../data/processed/X_test_num.pkl", "wb") as f:
    pickle.dump(X_test_num, f)

In [123]:
with open("../data/processed/X_train_cat.pkl", "wb") as f:
    pickle.dump(X_train_cat, f)

In [124]:
with open("../data/processed/X_test_cat.pkl", "wb") as f:
    pickle.dump(X_test_cat, f)

In [125]:
with open("../data/processed/X_train_text.pkl", "wb") as f:
    pickle.dump(X_train_text, f)

In [126]:
with open("../data/processed/X_test_text.pkl", "wb") as f:
    pickle.dump(X_test_text, f)

In [127]:
with open("../data/processed/X_train_seq.pkl", "wb") as f:
    pickle.dump(X_train_seq, f)

In [128]:
with open("../data/processed/X_test_seq.pkl", "wb") as f:
    pickle.dump(X_test_seq, f)

In [129]:
with open("../data/processed/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

In [130]:
with open("../data/processed/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)